In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append('../')

import pandas as pd
import os
import subprocess
import zipfile
import matplotlib.pyplot as plt
import lightgbm as lgb
import numpy as np

from itertools import product
from sklearn.model_selection import GroupShuffleSplit
from itertools import chain
from sklearn.model_selection import GroupKFold
from src.utils import * 
from src.feature_engineering import *
from src.pipeline import *
from src.run import *

In [ ]:
pd.reset_option('display.max_columns')
pd.set_option('display.max_columns', None)

In [ ]:
train = get_train_data()

In [ ]:
train = reduce_mem_usage(train)

In [ ]:
len(train.columns)

In [ ]:
train = drop_constant_columns(train)

In [ ]:
col_names = train.columns.tolist()
duration_cols = []
for col in col_names:
    if 'duration' in col:
        duration_cols.append(col)
duration_cols

In [ ]:
train[duration_cols].head()

In [ ]:
train = type_conversion(train)

In [ ]:
train_df = train.copy()

In [ ]:
col_names = train_df.columns.tolist()
col_names

In [ ]:
train_df.head()

In [ ]:
rest_datetime_cols = ['legs0_arrivalAt', 'legs0_departureAt', 'legs1_arrivalAt', 'legs1_departureAt']

In [ ]:
train_df = fix_datetime_columns(train_df, rest_datetime_cols)

In [ ]:
train_df = searchRoute(train_df)

In [ ]:
train_df['frequentFlyer_count'] = train_df['frequentFlyer'].str.split('/').str.len().fillna(0).astype('int8')
train_df['is_frequentFlyer'] = train_df['frequentFlyer'].str.len() > 0

In [ ]:
train_df[['frequentFlyer_count', 'is_frequentFlyer']].tail()

--- Baseline process ---

In [ ]:
train_df = basic_data_pipeline()

In [ ]:
mean_score, std_score, scores = get_cv_score(train_df, n_splits=3)